# Credit Score Classification - AutoML Pipeline
## Microsoft Team Data Science Process (TDSP) Template

**Project:** Credit Score Classification using AutoML  
**Date:** 2026-05-20  
**Version:** 2.0 (TDSP Format)

---

## Table of Contents
1. [Business Understanding](#1-business-understanding)
2. [Data Acquisition](#2-data-acquisition)
3. [Data Understanding](#3-data-understanding)
4. [Data Preparation](#4-data-preparation)
5. [Modeling](#5-modeling)
6. [Evaluation](#6-evaluation)
7. [Deployment](#7-deployment)

---

## 1. Business Understanding

### 1.1 Project Objectives
The goal of this project is to develop an automated machine learning solution for credit score classification. This system will:
- Classify customers into credit score categories (Poor, Standard, Good)
- Provide probability scores for risk assessment
- Enable data-driven lending decisions
- Reduce manual review time and costs

### 1.2 Success Criteria
- **Accuracy:** Achieve >70% classification accuracy
- **Precision:** Minimize false positives for high-risk categories
- **Recall:** Identify majority of poor credit scores
- **F1-Score:** Balanced performance across all classes
- **Deployment:** Model registered and ready for production use

### 1.3 Business Impact
- Improved risk assessment accuracy
- Faster loan approval process
- Reduced default rates
- Better customer segmentation

### 1.4 Key Stakeholders
- **Business:** Credit Risk Management Team
- **Technical:** Data Science & ML Engineering Teams
- **End Users:** Loan Officers & Credit Analysts

## 2. Data Acquisition

### 2.1 Data Sources
- **Source:** Delta Lake (../data/train_silver/)
- **Format:** Parquet files
- **Size:** 100,000 samples
- **Features:** 28 columns including demographic, financial, and credit history data

In [ ]:
from pathlib import Path

# Find project root based on known workspace markers
def find_project_root(start_path: Path = Path.cwd()):
    markers = {'data', 'outputs', 'notebooks', 'scripts'}
    for path in [start_path, *start_path.parents]:
        if any((path / marker).exists() for marker in markers):
            return path
    raise FileNotFoundError("Project root not found from current working directory")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'train_silver'
LOG_DIR = PROJECT_ROOT / 'outputs' / 'logs'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'predictions'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for directory in (LOG_DIR, OUTPUT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Configuration dictionary
CONFIG = {
    'base': {
        'project_root': '.',
        'data_dir': str(Path('data') / 'train_silver'),
        'log_dir': str(Path('outputs') / 'logs'),
        'output_dir': str(Path('outputs') / 'predictions'),
        'figure_dir': str(Path('outputs') / 'figures')
    },
    'data': {
        'path': str(Path('data') / 'train_silver'),
        'sample_size': 100000,
        'test_size': 0.2,
        'random_state': 42,
        'target_column': 'credit_score'
    },
    'model': {
        'name': 'credit_score_classifier',
        'experiment_name': 'credit_score_automl_experiment',
        'time_budget': 120,
        'max_iter': 20,
        'cv_folds': 3,
        'estimators': ['lgbm', 'xgboost', 'rf', 'extra_tree'],
        'metric': 'accuracy'
    },
    'feature_engineering': {
        'drop_columns': ['id', 'customer_id', 'ssn'],  # High cardinality, no predictive value
        'categorical_encoding': 'auto',  # Will use target encoding for high cardinality
        'handle_missing': True
    },
    'output': {
        'predictions_file': str(Path('outputs') / 'predictions' / 'predictions_improved.parquet'),
        'log_file': str(Path('outputs') / 'logs' / 'automl_improved.log')
    }
}

print("[OK] Configuration loaded successfully")
print(f"Project root: {CONFIG['base']['project_root']}")
print(f"Data path: {CONFIG['data']['path']}")
print(f"Model: {CONFIG['model']['name']}")
print(f"Time Budget: {CONFIG['model']['time_budget']} seconds")
print(f"Target: {CONFIG['data']['target_column']}")

In [ ]:
import logging
import warnings
import sys
from datetime import datetime
from pathlib import Path

# Setup logging
log_path = Path(CONFIG['output']['log_file'])
log_path.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_path),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

logger.info("="*50)
logger.info("AutoML Pipeline Started")
logger.info(f"Timestamp: {datetime.now()}")
logger.info("="*50)

In [ ]:
# Install required packages
%pip install -q scikit-learn==1.5.1 pandas numpy pyarrow flaml mlflow xgboost matplotlib seaborn category_encoders lightgbm
logger.info("All packages installed successfully")

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import re
from pathlib import Path
import glob

def load_data_optimized(data_path, sample_size=None):
    """
    Load data from Delta table using optimized pyarrow
    Handles Delta Lake directory structure by reading only parquet files
    """
    try:
        logger.info(f"Loading data from {data_path}...")
        
        # Check if path is a directory (Delta Lake format)
        path_obj = Path(data_path)
        
        if path_obj.is_dir():
            # Find all parquet files, excluding metadata directories
            parquet_files = [
                str(f) for f in path_obj.glob('*.parquet')
                if not any(x in str(f) for x in ['_delta_log', 'metadata', '_metadata'])
            ]
            
            if not parquet_files:
                raise FileNotFoundError(f"No parquet files found in {data_path}")
            
            logger.info(f"Found {len(parquet_files)} parquet files")
            
            # Read all parquet files and combine
            dfs = []
            for file in parquet_files:
                table = pq.read_table(file, use_threads=True)
                dfs.append(table.to_pandas())
            
            df = pd.concat(dfs, ignore_index=True)
        else:
            # Single parquet file
            table = pq.read_table(data_path, use_threads=True)
            df = table.to_pandas()
        
        logger.info(f"Data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
        
        # Sample if needed
        if sample_size and sample_size < len(df):
            df = df.sample(n=sample_size, random_state=CONFIG['data']['random_state'])
            logger.info(f"Sampled to {sample_size:,} rows")
        
        # Clean column names
        df = df.rename(columns=lambda c: re.sub('[^A-Za-z0-9_]+', '_', c))
        
        return df
        
    except FileNotFoundError:
        logger.error(f"Data file not found: {data_path}")
        raise
    except Exception as e:
        logger.error(f"Error loading data: {str(e)}")
        raise

# Load data
X = load_data_optimized(
    CONFIG['data']['path'],
    CONFIG['data']['sample_size']
)

# Display sample
display(X.head())
logger.info(f"Columns: {list(X.columns)}")

## 3. Data Understanding

### 3.1 Data Quality Assessment
Comprehensive validation of data integrity, completeness, and distribution.

In [ ]:
def validate_data(df, target_col):
    """
    Comprehensive data validation
    """
    logger.info("Running data quality checks...")
    
    # Check 1: Empty dataset
    assert df.shape[0] > 0, "Dataset is empty"
    logger.info(f"Dataset size: {df.shape[0]:,} rows")
    
    # Check 2: Target column exists
    assert target_col in df.columns, f"Target column '{target_col}' not found"
    logger.info(f"Target column '{target_col}' found")
    
    # Check 3: Target classes
    n_classes = df[target_col].nunique()
    classes = df[target_col].unique()
    logger.info(f"Target has {n_classes} classes: {classes}")
    
    # Check 4: Missing values
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    cols_with_missing = missing_pct[missing_pct > 0].sort_values(ascending=False)
    if len(cols_with_missing) > 0:
        logger.warning(f"Columns with missing values:\n{cols_with_missing}")
    else:
        logger.info("No missing values found")
    
    # Check 5: Duplicates
    n_duplicates = df.duplicated().sum()
    if n_duplicates > 0:
        logger.warning(f"Found {n_duplicates} duplicate rows")
    else:
        logger.info("No duplicate rows")
    
    # Check 6: Data types
    logger.info(f"Data types:\n{df.dtypes.value_counts()}")
    
    return True

# Run validation
target_col = re.sub('[^A-Za-z0-9_]+', '_', CONFIG['data']['target_column'])
validate_data(X, target_col)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Analyze class distribution
distribution = X[target_col].value_counts(normalize=True)
dominant_class_proportion = distribution.max()

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
distribution.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title("Class Distribution (Proportion)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Class")
ax1.set_ylabel("Proportion")
ax1.grid(axis='y', alpha=0.3)

# Pie chart
X[target_col].value_counts().plot(kind='pie', ax=ax2, autopct='%1.1f%%', startangle=90)
ax2.set_title("Class Distribution (Count)", fontsize=14, fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

# Check balance
if dominant_class_proportion > 0.8:
    logger.warning(f"Dataset is imbalanced. Dominant class: {dominant_class_proportion*100:.2f}%")
    logger.info("Consider using class weights or SMOTE for handling imbalance")
else:
    logger.info(f"Dataset is reasonably balanced. Max class proportion: {dominant_class_proportion*100:.2f}%")

## 4. Data Preparation

### 4.1 Feature Engineering
Creating derived features and preparing data for modeling.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

def engineer_features(df, target_col, config):
    """
    Advanced feature engineering with proper encoding
    """
    logger.info("Starting feature engineering...")
    df = df.copy()
    
    # 1. Drop unnecessary columns
    drop_cols = [col for col in config['feature_engineering']['drop_columns'] if col in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
        logger.info(f"Dropped columns: {drop_cols}")
    
    # 2. Create derived features
    logger.info("Creating derived features...")
    
    # Debt-to-income ratio
    if 'outstanding_debt' in df.columns and 'annual_income' in df.columns:
        df['debt_to_income_ratio'] = df['outstanding_debt'] / (df['annual_income'] + 1)
    
    # Credit utilization per card
    if 'credit_utilization_ratio' in df.columns and 'num_credit_card' in df.columns:
        df['utilization_per_card'] = df['credit_utilization_ratio'] / (df['num_credit_card'] + 1)
    
    # Monthly income to EMI ratio
    if 'monthly_inhand_salary' in df.columns and 'total_emi_per_month' in df.columns:
        df['income_to_emi_ratio'] = df['monthly_inhand_salary'] / (df['total_emi_per_month'] + 1)
    
    # Age groups
    if 'age' in df.columns:
        df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], 
                                 labels=['18-25', '26-35', '36-45', '46-55', '55+'])
    
    logger.info(f"Created {4} derived features")
    
    # 3. Identify column types
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)
    
    logger.info(f"Numeric features: {len(numeric_cols)}")
    logger.info(f"Categorical features: {len(categorical_cols)}")
    
    return df, numeric_cols, categorical_cols

# Apply feature engineering
X_engineered, numeric_features, categorical_features = engineer_features(X, target_col, CONFIG)

logger.info(f"Final feature count: {X_engineered.shape[1] - 1}")  # -1 for target
display(X_engineered.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
y = X_engineered[target_col]
X_features = X_engineered.drop(columns=[target_col])

# Stratified split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    y,
    test_size=CONFIG['data']['test_size'],
    random_state=CONFIG['data']['random_state'],
    stratify=y  # Maintain class distribution
)

logger.info(f"Train set: {X_train.shape[0]:,} samples")
logger.info(f"Test set: {X_test.shape[0]:,} samples")
logger.info(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")
logger.info(f"Test class distribution:\n{y_test.value_counts(normalize=True)}")

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import flaml

logger.info("Setting up preprocessing pipeline...")

# Convert dtypes
X_train = X_train.convert_dtypes()
X_test = X_test.convert_dtypes()

if flaml.__version__ > "2.3.6":
    X_train, _ = flaml.automl.data.auto_convert_dtypes_pandas(X_train)
    X_test, _ = flaml.automl.data.auto_convert_dtypes_pandas(X_test)

# Handle missing values
def create_preprocessor(X_train, numeric_features, categorical_features):
    """
    Create preprocessing pipeline
    """
    # Numeric pipeline
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
    ])
    
    # Categorical pipeline
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
    ])
    
    # Combine
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='passthrough'
    )
    
    return preprocessor

# Get actual column names from data
actual_numeric = [col for col in numeric_features if col in X_train.columns]
actual_categorical = [col for col in categorical_features if col in X_train.columns]

preprocessor = create_preprocessor(X_train, actual_numeric, actual_categorical)

# Fit and transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert back to DataFrame
feature_names = actual_numeric + actual_categorical + [col for col in X_train.columns if col not in actual_numeric + actual_categorical]
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

logger.info(f"Preprocessing complete")
logger.info(f"Train shape: {X_train_processed.shape}")
logger.info(f"Test shape: {X_test_processed.shape}")

## 5. Modeling

### 5.1 Model Selection & Training
Using FLAML AutoML for automated model selection and hyperparameter tuning.

In [ ]:
import mlflow
import os

# MLflow setup
MLFLOW_DIR = PROJECT_ROOT / 'outputs' / 'mlruns'
MLFLOW_DB = PROJECT_ROOT / 'outputs' / 'mlflow.db'
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")
os.environ['MLFLOW_ARTIFACT_URI'] = MLFLOW_DIR.resolve().as_uri()
mlflow.autolog(exclusive=False)
desired_artifact_root = MLFLOW_DIR.resolve().as_uri()
exp_name = CONFIG['model']['experiment_name']
exp = mlflow.get_experiment_by_name(exp_name)
if exp is None:
    mlflow.create_experiment(exp_name, artifact_location=desired_artifact_root)
elif not str(exp.artifact_location).startswith(desired_artifact_root):
    exp_name = f"{exp_name}_outputs"
    if mlflow.get_experiment_by_name(exp_name) is None:
        mlflow.create_experiment(exp_name, artifact_location=desired_artifact_root)
CONFIG['model']['experiment_name'] = exp_name
mlflow.set_experiment(exp_name)

logger.info(f"[OK] MLflow experiment: {CONFIG['model']['experiment_name']}")

In [ ]:
from flaml.automl.automl import AutoML

logger.info("="*50)
logger.info("Starting AutoML Training")
logger.info("="*50)

# End any active MLflow runs (important for notebook re-runs)
try:
    mlflow.end_run()
    logger.info("Ended previous MLflow run")
except Exception:
    pass

# Configure AutoML
os.environ["FLAML_MLFLOW_LOG_LATENCY"] = "1"

automl_settings = {
    "time_budget": CONFIG['model']['time_budget'],
    "task": "multiclass",
    "log_file_name": CONFIG['output']['log_file'].replace('.log', '_flaml.log'),
    "log_type": "better",
    "eval_method": "cv",
    "n_splits": CONFIG['model']['cv_folds'],
    "max_iter": CONFIG['model']['max_iter'],
    "force_cancel": True,
    "seed": CONFIG['data']['random_state'],
    "mlflow_exp_name": CONFIG['model']['experiment_name'],
    "use_spark": False,
    "verbose": 1,
    "estimator_list": CONFIG['model']['estimators'],
}

if flaml.__version__ > "2.3.3":
    automl_settings["entrypoint"] = "low-code"

# Create AutoML instance
automl = AutoML(**automl_settings)

# Train
with mlflow.start_run(run_name=CONFIG['model']['name']) as run:
    logger.info(f"Training with time budget: {CONFIG['model']['time_budget']}s")
    
    automl.fit(
        X_train=X_train_processed,
        y_train=y_train
    )
    
    logger.info("AutoML training completed")
    logger.info(f"Best estimator: {automl.best_estimator}")
    logger.info(f"Best config: {automl.best_config}")
    logger.info(f"Best loss: {automl.best_loss:.4f}")

## 6. Evaluation

### 6.1 Model Performance Assessment
Comprehensive evaluation using multiple metrics and visualizations.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
import seaborn as sns

logger.info("="*50)
logger.info("Model Evaluation")
logger.info("="*50)

# Make predictions
y_pred_train = automl.predict(X_train_processed)
y_pred_test = automl.predict(X_test_processed)
y_pred_proba = automl.predict_proba(X_test_processed)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test, average='weighted')
test_recall = recall_score(y_test, y_pred_test, average='weighted')
test_f1 = f1_score(y_test, y_pred_test, average='weighted')

# Print metrics
print("\n" + "="*50)
print("PERFORMANCE METRICS")
print("="*50)
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Precision:      {test_precision:.4f}")
print(f"Recall:         {test_recall:.4f}")
print(f"F1 Score:       {test_f1:.4f}")
print("="*50)

# Log to MLflow
mlflow.log_metrics({
    "train_accuracy": train_accuracy,
    "test_accuracy": test_accuracy,
    "precision": test_precision,
    "recall": test_recall,
    "f1_score": test_f1
})

# Classification report
print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_test))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), 
            yticklabels=sorted(y.unique()))
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
confusion_matrix_file = FIGURE_DIR / 'confusion_matrix.png'
plt.savefig(confusion_matrix_file, dpi=300, bbox_inches='tight')
mlflow.log_artifact(str(confusion_matrix_file))
plt.show()

logger.info("Model evaluation completed")

In [ ]:
# Feature importance
try:
    if hasattr(automl.model, 'feature_importances_'):
        importances = np.asarray(automl.model.feature_importances_).ravel()
        if hasattr(X_train_processed, 'columns'):
            feature_names = list(X_train_processed.columns)
        else:
            feature_names = [f'feature_{i}' for i in range(len(importances))]
        n = min(len(feature_names), len(importances))
        if len(feature_names) != len(importances):
            logger.warning(
                f"[WARN] Feature name and importance length mismatch ({len(feature_names)} vs {len(importances)}). "
                "Truncating to aligned length."
            )
        feature_importance = pd.DataFrame({
            'feature': feature_names[:n],
            'importance': importances[:n]
        }).sort_values('importance', ascending=False)
        
        # Plot top 20 features
        plt.figure(figsize=(12, 8))
        top_features = feature_importance.head(20)
        plt.barh(range(len(top_features)), top_features['importance'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Importance')
        plt.title('Top 20 Feature Importances', fontsize=16, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        feature_importance_file = FIGURE_DIR / 'feature_importance.png'
        plt.savefig(feature_importance_file, dpi=300, bbox_inches='tight')
        mlflow.log_artifact(str(feature_importance_file))
        plt.show()
        
        logger.info("[OK] Feature importance analysis completed")
        print("\nTop 10 Most Important Features:")
        print(feature_importance.head(10).to_string(index=False))
    else:
        logger.warning("[WARN] Feature importance not available for this model")
except Exception as e:
    logger.warning(f"[WARN] Could not extract feature importance: {str(e)}")

## 7. Deployment

### 7.1 Model Registration & Deployment
Registering the model to MLflow Model Registry for production deployment.

In [ ]:
# Register model
model_path = f"runs:/{run.info.run_id}/model"

try:
    registered_model = mlflow.register_model(
        model_uri=model_path,
        name=CONFIG['model']['name']
    )
    
    logger.info(f"Model '{registered_model.name}' version {registered_model.version} registered")
    print(f"\nModel registered successfully!")
    print(f"   Name: {registered_model.name}")
    print(f"   Version: {registered_model.version}")
    print(f"   Run ID: {run.info.run_id}")
    
except Exception as e:
    logger.error(f"Error registering model: {str(e)}")
    raise

In [ ]:
# Load model and generate predictions
model_uri = f"models:/{CONFIG['model']['name']}/{registered_model.version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Convert categorical to string for MLflow compatibility
X_test_for_pred = X_test_processed.copy()
for col in X_test_for_pred.select_dtypes(include=['category']).columns:
    X_test_for_pred[col] = X_test_for_pred[col].astype(str)

# Generate predictions
predictions = loaded_model.predict(X_test_for_pred)

# Map numeric predictions back to original labels
# The model returns numeric labels (0, 1, 2), we need to map them back to strings
label_mapping = {i: label for i, label in enumerate(sorted(y.unique()))}
predictions_mapped = [label_mapping.get(pred, pred) for pred in predictions]

# Create results DataFrame
results_df = X_test.copy()
results_df['actual'] = y_test.values
results_df['predicted'] = predictions_mapped
results_df['correct'] = (results_df['actual'] == results_df['predicted'])

# Add prediction probabilities
for i, class_name in enumerate(sorted(y.unique())):
    results_df[f'prob_{class_name}'] = y_pred_proba[:, i]

logger.info(f"Generated {len(predictions):,} predictions")
print(f"\nPrediction Summary:")
print(f"Total predictions: {len(predictions):,}")
print(f"Correct predictions: {results_df['correct'].sum():,} ({results_df['correct'].mean()*100:.2f}%)")
print(f"\nFirst 10 predictions:")
display(results_df[['actual', 'predicted', 'correct'] + [c for c in results_df.columns if c.startswith('prob_')]].head(10))

In [ ]:
from pathlib import Path
import os

# Save predictions using project-root anchored relative path
output_file = PROJECT_ROOT / CONFIG['output']['predictions_file']
output_file.parent.mkdir(parents=True, exist_ok=True)
results_df.to_parquet(output_file, index=False)

try:
    rel_output_file = output_file.resolve().relative_to(PROJECT_ROOT.resolve())
except Exception:
    rel_output_file = Path(CONFIG['output']['predictions_file'])

logger.info(f"[OK] Predictions saved to {rel_output_file.as_posix()}")
print("\n[OK] Predictions saved successfully!")
print(f"   File: {rel_output_file.as_posix()}")
print(f"   Location: {rel_output_file.as_posix()}")
print(f"   Size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")
print(f"   Rows: {len(results_df):,}")

In [ ]:
print("\n" + "="*60)
print("AUTOML PIPELINE SUMMARY")
print("="*60)
print("\nDataset:")
print(f"   Total samples: {len(X):,}")
print(f"   Train samples: {len(X_train):,}")
print(f"   Test samples: {len(X_test):,}")
print(f"   Features: {X_train_processed.shape[1]}")
print(f"   Target classes: {y.nunique()}")

print("\nModel:")
print(f"   Best estimator: {automl.best_estimator}")
print(f"   Training time: {CONFIG['model']['time_budget']}s")
print(f"   Iterations: {CONFIG['model']['max_iter']}")

print("\nPerformance:")
print(f"   Train Accuracy: {train_accuracy:.4f}")
print(f"   Test Accuracy:  {test_accuracy:.4f}")
print(f"   Precision:      {test_precision:.4f}")
print(f"   Recall:         {test_recall:.4f}")
print(f"   F1 Score:       {test_f1:.4f}")

print("\nOutputs:")
print(f"   Model: {CONFIG['model']['name']} v{registered_model.version}")
print(f"   Predictions: {output_file}")
print(f"   Logs: {CONFIG['output']['log_file']}")

print("\n" + "="*60)
print("[OK] PIPELINE COMPLETED SUCCESSFULLY")
print("="*60)

logger.info("="*50)
logger.info("AutoML Pipeline Completed Successfully")
logger.info(f"Timestamp: {datetime.now()}")
logger.info("="*50)

## Conclusion

### Project Summary
This TDSP-structured notebook demonstrates a complete machine learning pipeline for credit score classification:

**Key Achievements:**
- Structured approach following Microsoft TDSP methodology
- Automated model selection using FLAML
- Comprehensive evaluation and validation
- Production-ready model deployment

**Next Steps:**
1. Monitor model performance in production
2. Implement A/B testing framework
3. Schedule periodic model retraining
4. Gather feedback from stakeholders

---

**Documentation:**
- [Microsoft TDSP](https://docs.microsoft.com/en-us/azure/machine-learning/team-data-science-process/)
- [FLAML Documentation](https://microsoft.github.io/FLAML/)
- [MLflow Documentation](https://mlflow.org/)